# 85장 · FastAPI 앱 재시작과 DB 호환성

상태 경로가 정상인 새 앱에서 어제 만든 할 일을 조회해 봅니다.

새 앱에서 due_date와 priority를 사용하는데 DB에는 예전 열만 있는 상황을 점검합니다. 앱 교체 전후에 같은 ID를 조회해 데이터 보존과 호환성을 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 fastapi 0.135.0, pydantic 2.14.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


### 앱이 요구하는 스키마 이름

<p>새 앱이 필요로 하는 버전은 082_todo_v1과 084_todo_v2입니다. 적용 내역에 082_todo_v1만 넣고 차집합을 구하면 부족한 084_todo_v2 한 개가 출력됩니다.</p>


## 기존 DB 스키마와 앱 버전 확인

제공 구성의 서비스 이름은 db, migrate, api입니다. migrate는 설정할 때만 쓰며 API 시작 검사는 필요한 두 마이그레이션이 적용됐는지 읽습니다. 아래는 082·084 스키마 적용 여부를 확인하는 작은 기록 예제입니다. 파일 내용이 맞는지도 확인하려면 원본처럼 버전별 해시를 대조합니다.


### 누락된 스키마와 새 앱의 읽기 경로

<p>새 앱은 due_date와 priority를 조회합니다. 이전 DB에 열이 없다면 앱 프로세스를 새로 시작하는 것만으로는 두 열이 생기지 않습니다. 첫 셀로 필요한 버전을 확인하고, 다음 셀에서는 필요한 열을 갖춘 시험 DB로 동작을 비교합니다.</p>


### 셀 01 · 기존 DB 스키마 확인

실제 배포에서는 DB의 schema_migrations에서 적용 이력을 읽습니다. 여기서는 준비 기록을 받아 누락된 버전을 찾습니다.


In [1]:
required = {"082_todo_v1", "084_todo_v2"}
applied = {"082_todo_v1"}
print(sorted(required - applied))

['084_todo_v2']


084_todo_v2가 빠졌습니다. 이 상태에서 화면만 배포 완료로 표시하면 마감일을 저장하는 요청이 DB 경계에서 실패합니다. 운영 DB에 실제로 적용됐는지는 이 예시 기록으로 확인할 수 없습니다.


## 새 앱 인스턴스에서 저장값 재조회

수업용 FastAPI 앱 두 개가 같은 임시 SQLite 파일을 엽니다. 첫 앱에서 v2 필드를 가진 항목을 저장한 뒤, 새 앱의 HTTP 경로에서 같은 ID로 조회합니다. 실제 Compose 재배포와 PostgreSQL 대신 프로세스 안의 서비스·파일 경계를 확인합니다.


### 같은 파일을 사용하는 두 TestClient

<p>make_app(path)를 두 번 호출하면 FastAPI 앱은 두 개가 생깁니다. 두 앱 모두 release.db를 열도록 경로를 고정해, 첫 앱에서 쓴 값을 새 앱이 정말 다시 읽는지 확인합니다.</p>


### 셀 02 · 앱의 DB 경로와 조회 함수

이 축약본의 POST는 수업에서 정한 제목·마감일·우선순위 입력을 받습니다. 상태가 어디에 저장되는지 보세요.

앞의 배포 기록에서는 084가 빠졌으므로 migrate 단계로 돌아가야 합니다. 다음 셀은 필요한 열을 갖춘 새 SQLite 파일을 사용합니다. 여기에서 확인하는 일은 배포 환경의 이관이 아니라 저장한 값을 새 앱에서 다시 읽는 일입니다.


In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import closing
import sqlite3
from typing import Literal
from datetime import date
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, StrictBool

tmp = TemporaryDirectory()
path = Path(tmp.name) / "release.db"
with closing(sqlite3.connect(path)) as db:
    db.execute("CREATE TABLE todos(id INTEGER PRIMARY KEY, title TEXT, completed INTEGER DEFAULT 0, due_date TEXT, priority TEXT)")
    db.commit()

class TodoInput(BaseModel):
    title: str
    due_date: date | None = None
    priority: Literal["low", "normal", "high"] = "normal"

class CompleteInput(BaseModel):
    completed: StrictBool

def make_app(path):
    app = FastAPI()
    @app.post("/todos", status_code=201)
    def add(body: TodoInput):
        with closing(sqlite3.connect(path)) as db:
            cur = db.execute("INSERT INTO todos(title,due_date,priority) VALUES(?,?,?)",
                             (body.title, body.due_date.isoformat() if body.due_date else None, body.priority))
            db.commit()
            return {"data": {"id": cur.lastrowid, "title": body.title, "completed": False,
                             "due_date": body.due_date, "priority": body.priority}}
    @app.get("/todos/{id}")
    def read(id: int):
        with closing(sqlite3.connect(path)) as db:
            row = db.execute("SELECT id,title,completed,due_date,priority FROM todos WHERE id=?", (id,)).fetchone()
        if row is None:
            raise HTTPException(404)
        return {"data": {"id":row[0], "title":row[1], "completed":bool(row[2]), "due_date":row[3], "priority":row[4]}}
    @app.get("/health")
    def health():
        return {"application": "todo"}

    @app.patch("/todos/{id}")
    def complete(id: int, body: CompleteInput):
        with closing(sqlite3.connect(path)) as db:
            changed = db.execute("UPDATE todos SET completed=? WHERE id=?", (body.completed, id))
            if changed.rowcount == 0:
                raise HTTPException(404)
            db.commit()
        return read(id)

    @app.delete("/todos/{id}")
    def remove(id: int):
        with closing(sqlite3.connect(path)) as db:
            changed = db.execute("DELETE FROM todos WHERE id=?", (id,))
            if changed.rowcount == 0:
                raise HTTPException(404)
            db.commit()
        return {"id": id}

    return app

API 객체가 바뀌어도 연결할 파일 경로는 같습니다. 원본 배포 예제의 PostgreSQL 접속 설정과 스키마 검사, 전체 CRUD는 별도 구현에 있습니다.


#### 앱을 다시 띄운 뒤 무엇을 읽을까요?

<mark>새 앱 인스턴스가 같은 DB 파일에서 기존 행을 읽어야 재시작 뒤 보존을 확인할 수 있습니다.</mark> 등록 직후의 응답과 새 인스턴스에서 다시 읽은 제목·기한·우선순위를 짝지으세요. 같은 프로세스 메모리에 남은 값을 보는 검사와 구별됩니다.

선수 개념 · 84장 SQLite 스키마 변경과 부분 업데이트

### 앱 인스턴스를 바꾼 뒤 같은 ID가 조회되는 경로

두 TestClient는 서로 다른 FastAPI 앱을 사용합니다. 공유하는 것은 release.db 경로입니다.

1. 원래 앱 · 인스턴스 = original
2. POST /todos · 응답 상태 = 201
3. 파일 DB · 저장 파일 = release.db
4. 새 앱 · 인스턴스 = replacement
5. GET /todos/1 · 조회 기준 = baseline의 id
6. 레코드 대조 · baseline == observed = True


### 셀 03 · 새 앱에서 같은 ID 재조회

수업 앱 두 개를 차례로 만들고 실제 HTTP 응답 data를 비교합니다.


In [3]:
with TestClient(make_app(path)) as original:
    created = original.post("/todos", json={"title": "복습", "due_date": "2027-04-02", "priority": "high"})
    baseline = created.json()["data"]
with TestClient(make_app(path)) as replacement:
    observed = replacement.get(f"/todos/{baseline['id']}").json()["data"]
print(created.status_code, baseline == observed)
print(observed["title"], observed["due_date"], observed["priority"])

201 True
복습 2027-04-02 high


201은 첫 앱이 보낸 등록 응답입니다. True는 같은 파일을 사용하는 새 앱이 같은 ID에서 기존의 모든 값을 읽었다는 뜻입니다. 클라우드 배포나 컨테이너 재시작은 이 비교에 포함되지 않습니다.


#### 앱 인스턴스를 바꾼 뒤 같은 ID가 조회되는 경로

```mermaid
flowchart TD
nfirst["원래 앱"]
  nwrite["POST /todos"]
  ndb["파일 DB"]
  nsecond["새 앱"]
  nread["GET /todos/1"]
  ncompare["레코드 대조"]
  nfirst --> nwrite
  nwrite --> ndb
  ndb --> nsecond
  nsecond --> nread
  nread --> ncompare
```

두 TestClient는 서로 다른 FastAPI 앱을 사용합니다. 공유하는 것은 release.db 경로입니다.

### 기존 필드와 새 필드를 한 행에서 비교하기

<p>응답 코드 201은 첫 요청의 등록 성공입니다. 그 옆의 True는 저장 직후 baseline과 새 앱에서 조회한 observed의 전체 레코드가 같다는 뜻입니다. 제목·기한·우선순위 출력에서 무엇이 유지됐는지도 확인하세요.</p>


### 과제 04 · 이전 스키마의 열로 같은 행 조회

v2 앱으로 저장한 항목을 같은 DB에서 이전 조회문으로 다시 읽으세요. v1이 쓰던 세 열을 먼저 출력하고 v2의 날짜와 우선순위도 여전히 남았는지 확인하세요.


In [4]:
# path에 다시 연결해 같은 ID로 v1의 세 열과 v2에 추가된 두 열을 각각 조회하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>with closing(sqlite3.connect(path)) as db:
    key = (baseline[&quot;id&quot;],)
    legacy = db.execute(&quot;SELECT id, title, completed FROM todos WHERE id=?&quot;, key).fetchone()
    extra = db.execute(&quot;SELECT due_date, priority FROM todos WHERE id=?&quot;, key).fetchone()
print(legacy)
print(extra)</code></pre>
<p>실행 결과</p><pre><code>(1, &#x27;복습&#x27;, 0)
(&#x27;2027-04-02&#x27;, &#x27;high&#x27;)</code></pre>
<p>이전 형식의 SELECT는 필요한 세 열만 이름으로 지정하므로 열이 늘어난 DB에서도 기존 행을 읽습니다. 두 번째 SELECT는 새 데이터가 남았는지 확인합니다. 여기서 시험한 범위는 읽기 호환성이며 이전 앱의 쓰기나 이미지 롤백은 별도 검증이 필요합니다.</p>
</details>


### 과제 05 · 교체한 로컬 앱의 출시 전 기능 검사

두 번째 앱 인스턴스에 연결해 health, 기존 레코드, 새 할 일의 등록·완료·삭제를 확인하세요. 테스트용으로 만든 ID만 삭제하고 원래 ‘복습’ 행은 끝까지 남겨야 합니다.


In [ ]:
with TestClient(make_app(path)) as probe:
    # 기존 ID의 응답을 baseline과 대조하세요.
    # 테스트용 항목을 등록하고 그 ID만 완료·삭제한 뒤 다시 조회하세요.
    pass

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>with TestClient(make_app(path)) as probe:
    status = probe.get(&quot;/health&quot;)
    assert status.status_code == 200 and status.json()[&quot;application&quot;] == &quot;todo&quot;
    original_url = f&quot;/todos/{baseline[&#x27;id&#x27;]}&quot;
    assert probe.get(original_url).json()[&quot;data&quot;] == baseline
    new = probe.post(&quot;/todos&quot;, json={&quot;title&quot;: &quot;출시 점검&quot;})
    assert new.status_code == 201
    test_url = f&quot;/todos/{new.json()[&#x27;data&#x27;][&#x27;id&#x27;]}&quot;
    changed = probe.patch(test_url, json={&quot;completed&quot;: True})
    assert changed.status_code == 200
    assert probe.get(test_url).json()[&quot;data&quot;][&quot;completed&quot;] is True
    removed = probe.delete(test_url)
    print(status.status_code, new.status_code, changed.status_code, removed.status_code)
    print(probe.get(test_url).status_code, probe.get(original_url).json()[&quot;data&quot;] == baseline)</code></pre>
<p>실행 결과</p><pre><code>200 201 200 200
404 True</code></pre>
<p>첫 줄은 health, 등록, 수정, 삭제 응답입니다. 두 번째 줄은 지운 테스트 항목만 사라졌고 원래 레코드는 모든 필드가 그대로임을 보여 줍니다. health만 검사했다면 쓰기 경로의 결함이나 데이터 손실을 발견할 수 없습니다. 이 실습은 임시 SQLite 파일과 로컬 TestClient에서 수행합니다.</p>
</details>


## 이전 버전 DB로 새 앱을 띄운 뒤 같은 ID 복구하기

빈 DB로 실행한 CRUD가 모두 통과해도 기존 데이터가 들어 있는 환경에서는 실패할 수 있습니다. 예전 스키마에 할 일 한 건을 담아 실제로 새 앱의 상세 조회를 호출하고, 스키마를 고친 뒤 같은 요청을 다시 보냅니다.

### 셀 06 · 상태 경로는 200인데 상세 조회는 실패하는 상황

셀 02의 make_app과 TestClient를 준비하세요. 기존 DB와 별도의 임시 파일을 준비합니다. 82장 방식의 세 열만 만들고, 앞 셀의 make_app으로 마감일·우선순위를 읽는 앱을 실행합니다.

In [5]:
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import closing
import sqlite3

legacy_space = TemporaryDirectory()
legacy_path = Path(legacy_space.name) / "old.db"
with closing(sqlite3.connect(legacy_path)) as old:
    old.execute("CREATE TABLE todos(id INTEGER PRIMARY KEY, title TEXT, completed INTEGER DEFAULT 0)")
    old.execute("INSERT INTO todos VALUES (7, '기존 과제', 1)")
    old.commit()

with TestClient(make_app(legacy_path), raise_server_exceptions=False) as probe:
    print(probe.get("/health").status_code, probe.get("/todos/7").status_code)

200 500


200 500이 함께 나옵니다. health는 DB를 읽지 않지만 상세 조회의 SELECT는 기존 테이블에 없는 열을 참조합니다. raise_server_exceptions=False로 서버 예외를 클라이언트가 실제로 받는 500 응답으로 확인했습니다.

### 과제 07 · 기존 행을 보존하는 마이그레이션 검산하기

임시 DB에 due_date와 priority를 추가하세요. 기존 행의 우선순위 기본값은 normal입니다. 같은 노트북에서 연습 셀을 다시 실행해도 중복 열 오류가 나지 않게 하고, 새 앱으로 ID 7의 다섯 값을 출력하세요.

In [6]:
# PRAGMA table_info(todos)에서 기존 열 이름을 확인하세요.
# 없는 열만 추가한 뒤 make_app(legacy_path)로 같은 ID를 조회하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>with closing(sqlite3.connect(legacy_path)) as old:
    columns = {row[1] for row in old.execute(&quot;PRAGMA table_info(todos)&quot;)}
    if &quot;due_date&quot; not in columns:
        old.execute(&quot;ALTER TABLE todos ADD COLUMN due_date TEXT&quot;)
    if &quot;priority&quot; not in columns:
        old.execute(&quot;ALTER TABLE todos ADD COLUMN priority TEXT NOT NULL DEFAULT &#x27;normal&#x27;&quot;)
    old.commit()

with TestClient(make_app(legacy_path)) as probe:
    recovered = probe.get(&quot;/todos/7&quot;)
data = recovered.json()[&quot;data&quot;]
print(recovered.status_code, data[&quot;id&quot;], data[&quot;title&quot;], data[&quot;completed&quot;], data[&quot;due_date&quot;], data[&quot;priority&quot;])</code></pre>
<p>실행 결과</p><pre><code>200 7 기존 과제 True None normal</code></pre>
<p>200 7 기존 과제 True None normal이 나옵니다. 예전 제목과 완료 여부가 그대로이고 새 열에는 정한 기본값이 들어갔습니다. 이 실행은 로컬 SQLite 복사본에서 이뤄졌습니다. PostgreSQL 운영 마이그레이션과 AWS 전환은 해당 수업 환경에서 별도로 검증합니다.</p>
</details>

<mark>배포 전부터 있던 행을 새 버전에서 다시 읽으세요.</mark> 새 항목만 만들어 검사하면 기존 세 열을 가진 DB와 새 코드의 호환성을 놓칩니다. 7번 행의 다섯 값을 기준으로 남겨야 이 실패가 복구됐는지 동료도 검사할 수 있습니다.

최종 프로젝트의 첫 인계 자료에는 기존 버전에서 만든 입력 하나, 새 버전 조회 결과, 아직 시험하지 않은 배포 환경을 함께 남겨 보세요.

## 돌아보기

새 앱이 켜졌다는 사실과 기존 할 일을 읽을 수 있다는 사실은 별개입니다. 같은 파일의 같은 ID로 제목·완료 상태·마감일을 다시 읽은 기록을 남기면 연결 경로가 달라져 빈 DB를 연 경우도 구별할 수 있습니다.
